### Setup: load the cleaned corpus
The same cleaned corpus and the same settings as Member 1 (from `representations.py`).

In [ ]:
import os
import time

import numpy as np
import pandas as pd

from representations import (
    load_cleaned_corpus, load_tokenized_reviews,
    COUNT_SETTINGS, EMBEDDING_SETTINGS, MODELS_DIR,
)

corpus_df = load_cleaned_corpus()
tokenized_reviews = load_tokenized_reviews(corpus_df)
print("Reviews:", len(corpus_df), "| Tokens:", sum(len(t) for t in tokenized_reviews))

## Step 6: Word2Vec – CBOW (B1)

### 6.1 Hyperparameters

In [29]:
from gensim.models import Word2Vec, FastText
from representations import FASTTEXT_SETTINGS

for name, value in EMBEDDING_SETTINGS.items():
    print(f"{name:12} {value}")

vector_size  100
window       5
min_count    5
epochs       10
workers      4
seed         42


### 6.2 Train the CBOW model (sg=0)

In [30]:
start_time = time.time()
cbow_model = Word2Vec(sentences=tokenized_reviews, sg=0, **EMBEDDING_SETTINGS)

print(f"Training time: {time.time() - start_time:.1f} seconds")
print("Vocabulary size:", len(cbow_model.wv))
print("Vector size:    ", cbow_model.wv.vector_size)

Training time: 107.1 seconds
Vocabulary size: 35853
Vector size:     100


## Step 7: Word2Vec – Skip-gram (B2)

### 7.1 Train the Skip-gram model (sg=1, same hyperparameters)

In [31]:
start_time = time.time()
skipgram_model = Word2Vec(sentences=tokenized_reviews, sg=1, **EMBEDDING_SETTINGS)

print(f"Training time: {time.time() - start_time:.1f} seconds")
print("Vocabulary size:", len(skipgram_model.wv))
print("Vector size:    ", skipgram_model.wv.vector_size)

Training time: 301.5 seconds
Vocabulary size: 35853
Vector size:     100


### 7.2 Check: only the training scheme differs

In [32]:
hyperparameters = pd.DataFrame({
    "CBOW": [cbow_model.sg, cbow_model.vector_size, cbow_model.window, cbow_model.min_count, cbow_model.epochs],
    "Skip-gram": [skipgram_model.sg, skipgram_model.vector_size, skipgram_model.window, skipgram_model.min_count, skipgram_model.epochs],
}, index=["sg", "vector_size", "window", "min_count", "epochs"])
print(hyperparameters.to_string())

             CBOW  Skip-gram
sg              0          1
vector_size   100        100
window          5          5
min_count       5          5
epochs         10         10


## Step 9: fastText (B4)

### 9.1 Train the fastText model

In [33]:
start_time = time.time()
fasttext_model = FastText(sentences=tokenized_reviews, **EMBEDDING_SETTINGS, **FASTTEXT_SETTINGS)

print(f"Training time: {time.time() - start_time:.1f} seconds")
print("Vocabulary size:", len(fasttext_model.wv))
print("Character n-grams:", FASTTEXT_SETTINGS["min_n"], "to", FASTTEXT_SETTINGS["max_n"])

Training time: 414.9 seconds
Vocabulary size: 35853
Character n-grams: 3 to 6


### 9.3 Out-of-vocabulary / misspelled words in Word2Vec vs fastText

In [34]:
oov_words = ["moviee", "terrrible", "awesomee", "boringgg"]

for word in oov_words:
    print(f"Query: '{word}'")
    print("  In Word2Vec vocabulary:", word in cbow_model.wv.key_to_index)
    try:
        print("  Word2Vec (CBOW):", cbow_model.wv.most_similar(word, topn=5))
    except KeyError as error:
        print("  Word2Vec (CBOW): ERROR ->", error)
    print("  In fastText vocabulary:", word in fasttext_model.wv.key_to_index)
    neighbours = fasttext_model.wv.most_similar(word, topn=5)
    print("  fastText:", [(w, round(s, 3)) for w, s in neighbours])
    print()

Query: 'moviee'
  In Word2Vec vocabulary: False
  Word2Vec (CBOW): ERROR -> "Key 'moviee' not present in vocabulary"
  In fastText vocabulary: False
  fastText: [('movie-i', 0.96), ('movie-a', 0.954), ('movie-it', 0.942), ('movie.i', 0.939), ('movies.i', 0.936)]

Query: 'terrrible'
  In Word2Vec vocabulary: False
  Word2Vec (CBOW): ERROR -> "Key 'terrrible' not present in vocabulary"
  In fastText vocabulary: False
  fastText: [('terrible', 0.971), ('horrible', 0.919), ('awful', 0.81), ('terribleness', 0.803), ('atrocious', 0.802)]

Query: 'awesomee'
  In Word2Vec vocabulary: False
  Word2Vec (CBOW): ERROR -> "Key 'awesomee' not present in vocabulary"
  In fastText vocabulary: False
  fastText: [('awesome', 0.929), ('awesomely', 0.705), ('amazing', 0.689), ('amaze', 0.671), ('fantastic', 0.671)]

Query: 'boringgg'
  In Word2Vec vocabulary: False
  Word2Vec (CBOW): ERROR -> "Key 'boringgg' not present in vocabulary"
  In fastText vocabulary: False
  fastText: [('boring', 0.956), ('bore'